# Busca de papers no arXiv sobre modelos de difusão quânticos

Este notebook é uma versão adaptada dos códigos de busca enviados no `.zip`, mas focada apenas no **arXiv** e em temas ligados a **quantum diffusion models**, incluindo:

- quantum diffusion models;
- quantum denoising diffusion models;
- quantum score-based generative models;
- stochastic Schrödinger diffusion models;
- diffusion models aplicados a quantum generative learning;
- quantum circuits / QNNs usados em modelos generativos de difusão.

A busca usa a API pública do arXiv, então **não precisa de chave de API**.


## 1. Instalação e importações

Execute esta célula uma vez no ambiente do notebook.


In [ ]:
# Instala automaticamente pacotes ausentes em ambientes limpos.
import importlib.util
import subprocess
import sys

_REQUIRED_PACKAGES = {
    "feedparser": "feedparser",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
    "tqdm": "tqdm",
    "requests": "requests",
}

_missing = [pip_name for import_name, pip_name in _REQUIRED_PACKAGES.items()
            if importlib.util.find_spec(import_name) is None]

if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import re
import time
import urllib.parse
from pathlib import Path
from typing import Iterable, List, Dict, Any

import feedparser
import pandas as pd
import requests
from tqdm.auto import tqdm


## 2. Configuração geral

A ideia é usar várias subconsultas menores em vez de uma query gigante. Isso melhora a estabilidade da API do arXiv e facilita deduplicar os resultados depois.


In [ ]:
# =========================
# Configuração
# =========================

OUTPUT_DIR = Path("arxiv_quantum_diffusion_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# O arXiv recomenda uso educado da API. Evite valores muito baixos.
BATCH_SIZE = 100
SLEEP_SECONDS = 3.0
MAX_RESULTS_PER_QUERY = 300

# Use None para não filtrar por ano.
MIN_YEAR = 2020

# Pontuação mínima para aparecer no arquivo filtrado.
# Reduza para 0 se quiser inspecionar tudo manualmente.
MIN_RELEVANCE_SCORE = 6

# Ordenação final: "published_desc", "score_desc" ou "title".
SORT_MODE = "published_desc"


## 3. Queries para o arXiv

As consultas abaixo combinam termos de difusão, modelos generativos e computação quântica. Elas são propositalmente redundantes, pois depois o notebook remove duplicatas pelo identificador do arXiv.


In [ ]:
ARXIV_QUERIES = [
    'all:"quantum diffusion model"',
    'all:"quantum diffusion models"',
    'all:"quantum denoising diffusion"',
    'all:"quantum score-based"',
    'all:"score-based quantum"',
    'all:"stochastic Schrödinger diffusion"',
    'all:"stochastic Schrodinger diffusion"',
    'all:"quantum generative model" AND all:"diffusion"',
    'all:"quantum generative models" AND all:"diffusion"',
    'all:"quantum circuit" AND all:"diffusion model"',
    'all:"quantum neural network" AND all:"diffusion model"',
    'all:"variational quantum" AND all:"diffusion model"',
    'all:"quantum machine learning" AND all:"diffusion model"',
    'all:"quantum" AND all:"denoising diffusion probabilistic model"',
    'all:"quantum" AND all:"DDPM"',
    'all:"quantum" AND all:"score matching" AND all:"diffusion"',
    'all:"quantum" AND all:"Schrödinger bridge" AND all:"diffusion"',
    'all:"quantum" AND all:"Schrodinger bridge" AND all:"diffusion"',
]

len(ARXIV_QUERIES)


18

## 4. Funções auxiliares

Estas funções fazem a normalização de texto, extração de metadados, busca paginada, deduplicação e cálculo de uma pontuação simples de relevância.


In [ ]:
def normalize_text(value: Any) -> str:
    """Remove quebras de linha e espaços repetidos."""
    if not value:
        return ""
    return " ".join(str(value).split())


def strip_arxiv_version(arxiv_id: str) -> str:
    """Remove a versão do ID do arXiv, por exemplo 2506.19270v1 -> 2506.19270."""
    return re.sub(r"v\d+$", "", str(arxiv_id).strip())


def extract_year(date_string: str):
    """Extrai o ano de strings como '2025-06-10T...'"""
    if not date_string:
        return None
    match = re.search(r"(19|20)\d{2}", str(date_string))
    return int(match.group(0)) if match else None


def safe_filename(text: str, max_len: int = 120) -> str:
    text = normalize_text(text)
    text = re.sub(r"[^\w\-. ]+", "", text, flags=re.UNICODE)
    text = text.replace(" ", "_")
    return text[:max_len].strip("_") or "paper"


def entry_to_row(entry, search_query: str) -> Dict[str, Any]:
    arxiv_id = entry.id.split("/abs/")[-1]
    arxiv_id_base = strip_arxiv_version(arxiv_id)

    categories = []
    if hasattr(entry, "tags"):
        categories = [tag.get("term", "") for tag in entry.tags]

    pdf_url = ""
    for link in getattr(entry, "links", []):
        if link.get("type") == "application/pdf":
            pdf_url = link.get("href", "")
            break

    primary = getattr(entry, "arxiv_primary_category", {}) or {}
    if isinstance(primary, dict):
        primary_category = primary.get("term", "")
    else:
        primary_category = getattr(primary, "term", "")

    return {
        "search_query": search_query,
        "arxiv_id": arxiv_id,
        "arxiv_id_base": arxiv_id_base,
        "title": normalize_text(getattr(entry, "title", "")),
        "authors": ", ".join(a.name for a in getattr(entry, "authors", [])),
        "published": getattr(entry, "published", ""),
        "updated": getattr(entry, "updated", ""),
        "year": extract_year(getattr(entry, "published", "")),
        "primary_category": primary_category,
        "categories": ", ".join(categories),
        "summary": normalize_text(getattr(entry, "summary", "")),
        "comment": normalize_text(getattr(entry, "arxiv_comment", "")),
        "journal_ref": normalize_text(getattr(entry, "arxiv_journal_ref", "")),
        "doi": normalize_text(getattr(entry, "arxiv_doi", "")),
        "abs_url": getattr(entry, "link", ""),
        "pdf_url": pdf_url or f"https://arxiv.org/pdf/{arxiv_id_base}",
    }


In [ ]:
POSITIVE_PATTERNS = {
    r"\bdiffusion model(s)?\b": 6,
    r"\bdenoising diffusion\b": 7,
    r"\bdenoising diffusion probabilistic model(s)?\b": 8,
    r"\bddpm\b": 6,
    r"\bscore[- ]based\b": 6,
    r"\bscore matching\b": 5,
    r"\bgenerative model(s)?\b": 4,
    r"\bquantum generative\b": 6,
    r"\bquantum machine learning\b": 4,
    r"\bquantum neural network(s)?\b": 4,
    r"\bquantum circuit(s)?\b": 3,
    r"\bvariational quantum\b": 3,
    r"\bstochastic schr[oö]dinger diffusion\b": 8,
    r"\bschr[oö]dinger bridge\b": 4,
    r"\bdensity matrix\b": 2,
    r"\bopen quantum system(s)?\b": 2,
}

NEGATIVE_PATTERNS = {
    # Termos que frequentemente indicam difusão física/condensada, não modelo generativo de difusão.
    r"\bspin diffusion\b": -5,
    r"\bcharge diffusion\b": -5,
    r"\bthermal diffusion\b": -5,
    r"\bneutron diffusion\b": -5,
    r"\bparticle diffusion\b": -4,
    r"\bsubdiffusion\b": -3,
    r"\banderson localization\b": -3,
    r"\bquantum walk(s)?\b": -2,
}


def relevance_score(title: str, summary: str, categories: str = "") -> int:
    text = f"{title} {summary} {categories}".lower()
    score = 0

    # Requisito básico: presença de quantum e diffusion ajuda a evitar falsos positivos.
    if "quantum" in text:
        score += 3
    if "diffusion" in text:
        score += 3

    for pattern, weight in POSITIVE_PATTERNS.items():
        if re.search(pattern, text):
            score += weight

    for pattern, penalty in NEGATIVE_PATTERNS.items():
        if re.search(pattern, text):
            score += penalty

    return score


def classify_topic(title: str, summary: str) -> str:
    text = f"{title} {summary}".lower()

    if re.search(r"denoising diffusion|ddpm", text):
        return "DDPM / denoising diffusion"
    if re.search(r"score[- ]based|score matching", text):
        return "score-based diffusion"
    if re.search(r"stochastic schr[oö]dinger diffusion", text):
        return "stochastic Schrödinger diffusion"
    if re.search(r"schr[oö]dinger bridge", text):
        return "Schrödinger bridge"
    if re.search(r"quantum circuit|variational quantum|quantum neural network", text):
        return "quantum circuit / QNN generative model"
    if re.search(r"quantum generative", text):
        return "quantum generative model"
    return "other / needs manual check"


## 5. Busca no arXiv

Esta célula consulta o arXiv, respeitando intervalo entre requisições. O resultado bruto é salvo em CSV e XLSX.


In [ ]:
def fetch_arxiv_query(
    query: str,
    batch_size: int = BATCH_SIZE,
    max_results: int = MAX_RESULTS_PER_QUERY,
    sleep_seconds: float = SLEEP_SECONDS,
) -> List[Dict[str, Any]]:
    """Busca uma query no arXiv usando paginação."""
    base_url = "https://export.arxiv.org/api/query?"
    encoded_query = urllib.parse.quote(query)
    start = 0
    rows: List[Dict[str, Any]] = []

    while start < max_results:
        current_batch = min(batch_size, max_results - start)
        url = (
            f"{base_url}search_query={encoded_query}"
            f"&start={start}"
            f"&max_results={current_batch}"
            f"&sortBy=submittedDate"
            f"&sortOrder=descending"
        )

        feed = feedparser.parse(url)

        if getattr(feed, "bozo", False):
            print(f"[aviso] Possível problema ao ler feed da query: {query}")
            print(getattr(feed, "bozo_exception", ""))

        entries = getattr(feed, "entries", [])
        if not entries:
            break

        for entry in entries:
            rows.append(entry_to_row(entry, search_query=query))

        start += len(entries)
        if len(entries) < current_batch:
            break

        time.sleep(sleep_seconds)

    return rows


def search_arxiv_quantum_diffusion(queries: Iterable[str] = ARXIV_QUERIES) -> pd.DataFrame:
    all_rows: List[Dict[str, Any]] = []

    for query in tqdm(list(queries), desc="Consultando arXiv"):
        try:
            rows = fetch_arxiv_query(query)
            print(f"[arxiv] {len(rows):4d} resultados | {query}")
            all_rows.extend(rows)
        except Exception as exc:
            print(f"[erro] Query ignorada: {query}")
            print(exc)

    df = pd.DataFrame(all_rows)

    if df.empty:
        return df

    # Deduplicação por ID base, ignorando versões v1, v2 etc.
    df = df.drop_duplicates(subset=["arxiv_id_base"]).copy()

    # Filtro de ano opcional.
    if MIN_YEAR is not None:
        df = df[df["year"].fillna(0).astype(int) >= int(MIN_YEAR)].copy()

    # Pontuação e classe temática.
    df["relevance_score"] = df.apply(
        lambda row: relevance_score(row["title"], row["summary"], row.get("categories", "")),
        axis=1,
    )
    df["topic_class"] = df.apply(lambda row: classify_topic(row["title"], row["summary"]), axis=1)

    if SORT_MODE == "score_desc":
        df = df.sort_values(["relevance_score", "published"], ascending=[False, False])
    elif SORT_MODE == "title":
        df = df.sort_values("title", ascending=True)
    else:
        df = df.sort_values("published", ascending=False)

    return df.reset_index(drop=True)


df_all = search_arxiv_quantum_diffusion()

csv_all = OUTPUT_DIR / "arxiv_quantum_diffusion_all.csv"
xlsx_all = OUTPUT_DIR / "arxiv_quantum_diffusion_all.xlsx"

df_all.to_csv(csv_all, index=False)
df_all.to_excel(xlsx_all, index=False)

print(f"Total bruto deduplicado: {len(df_all)}")
print(f"CSV salvo em: {csv_all}")
print(f"XLSX salvo em: {xlsx_all}")

df_all.head(10)


Consultando arXiv:   0%|          | 0/18 [00:00<?, ?it/s]

[arxiv]   27 resultados | all:"quantum diffusion model"
[arxiv]   27 resultados | all:"quantum diffusion models"
[arxiv]    8 resultados | all:"quantum denoising diffusion"
[arxiv]    0 resultados | all:"quantum score-based"
[arxiv]    0 resultados | all:"score-based quantum"
[arxiv]    1 resultados | all:"stochastic Schrödinger diffusion"
[arxiv]    0 resultados | all:"stochastic Schrodinger diffusion"
[arxiv]   10 resultados | all:"quantum generative model" AND all:"diffusion"
[arxiv]   10 resultados | all:"quantum generative models" AND all:"diffusion"
[arxiv]   19 resultados | all:"quantum circuit" AND all:"diffusion model"
[arxiv]    2 resultados | all:"quantum neural network" AND all:"diffusion model"
[arxiv]    8 resultados | all:"variational quantum" AND all:"diffusion model"
[arxiv]    9 resultados | all:"quantum machine learning" AND all:"diffusion model"
[arxiv]   10 resultados | all:"quantum" AND all:"denoising diffusion probabilistic model"
[arxiv]    4 resultados | all:"q

,search_query,arxiv_id,arxiv_id_base,title,authors,published,updated,year,primary_category,categories,summary,comment,journal_ref,doi,abs_url,pdf_url,relevance_score,topic_class
0,"all:""quantum"" AND all:""score matching"" AND all...",2606.05217v1,2606.05217,The Score Hamiltonian: Mapping Diffusion Model...,"Peter Halmos, Boris Hanin",2026-05-28T18:39:52Z,2026-05-28T18:39:52Z,2026,math-ph,"math-ph, cs.AI, cs.LG, physics.data-an",We exhibit an exact correspondence between sam...,,,,https://arxiv.org/abs/2606.05217v1,https://arxiv.org/pdf/2606.05217v1,18,score-based diffusion
1,"all:""quantum circuit"" AND all:""diffusion model""",2605.23403v1,2605.23403,Hybrid Quantum-Classical Corrective Diffusion ...,"Rui Wang, Edoardo Pasetto, Amer Delilbasic, Mo...",2026-05-22T09:14:25Z,2026-05-22T09:14:25Z,2026,cs.LG,"cs.LG, physics.ao-ph, quant-ph",Statistical downscaling is a crucial component...,"11 pages, 9 figures. Submitted to IEEE QCE 2026",,,https://arxiv.org/abs/2605.23403v1,https://arxiv.org/pdf/2605.23403v1,18,quantum circuit / QNN generative model
2,"all:""quantum"" AND all:""denoising diffusion pro...",2605.19754v1,2605.19754,Hypothesis Tests for Observing Quantum Entangl...,"Vincent Alexander Croft, Lennart Voelz, Andrii...",2026-05-19T12:25:10Z,2026-05-19T12:25:10Z,2026,hep-ex,hep-ex,We present a novel experimental strategy for t...,,,,https://arxiv.org/abs/2605.19754v1,https://arxiv.org/pdf/2605.19754v1,21,DDPM / denoising diffusion
3,"all:""quantum"" AND all:""DDPM""",2605.13268v2,2605.13268,Physics Guided Generative Optimization for Tro...,WenBin Yan,2026-05-13T09:48:33Z,2026-06-04T20:01:09Z,2026,quant-ph,"quant-ph, cs.LG",Trotter Suzuki product formulas are the standa...,,,,https://arxiv.org/abs/2605.13268v2,https://arxiv.org/pdf/2605.13268v2,18,DDPM / denoising diffusion
4,"all:""stochastic Schrödinger diffusion""",2605.03573v3,2605.03573,Stochastic Schrödinger Diffusion Models for Pu...,"Jian Xu, Wei Chen, Shigui Li, Chao Li, Jingyua...",2026-05-05T09:44:14Z,2026-06-15T12:42:57Z,2026,stat.ML,"stat.ML, cs.LG",Quantum machine learning increasingly relies o...,,,,https://arxiv.org/abs/2605.03573v3,https://arxiv.org/pdf/2605.03573v3,34,score-based diffusion
5,"all:""quantum circuit"" AND all:""diffusion model""",2605.01367v1,2605.01367,From Characterization To Construction: Generat...,"King Yiu Yu, Aritra Sarkar, Erbing Hua, Maximi...",2026-05-02T10:25:10Z,2026-05-02T10:25:10Z,2026,quant-ph,"quant-ph, cs.LG, eess.SY",High-fidelity circuit execution on noisy inter...,"19 pages, 3 figures",,,https://arxiv.org/abs/2605.01367v1,https://arxiv.org/pdf/2605.01367v1,19,quantum circuit / QNN generative model
6,"all:""quantum"" AND all:""score matching"" AND all...",2604.21210v1,2604.21210,The Feedback Hamiltonian is the Score Function...,"Sagar Dubey, Alan John",2026-04-23T02:02:13Z,2026-04-23T02:02:13Z,2026,quant-ph,"quant-ph, cs.LG","In continuously monitored quantum systems, the...",14 pages,,,https://arxiv.org/abs/2604.21210v1,https://arxiv.org/pdf/2604.21210v1,23,score-based diffusion
7,"all:""quantum diffusion model""",2604.05612v2,2604.05612,Deuteron normalization and channel-dependent f...,"Byung-Geel Yu, Kook-Jin Kong, Tae Keun Choi",2026-04-07T09:02:56Z,2026-04-09T15:44:56Z,2026,hep-ph,"hep-ph, nucl-th",A combined view of the Jefferson Lab data on n...,"3 pages, 1 figure",,,https://arxiv.org/abs/2604.05612v2,https://arxiv.org/pdf/2604.05612v2,12,other / needs manual check
8,"all:""quantum generative model"" AND all:""diffus...",2604.01197v4,2604.01197,Learning and Generating Mixed States Prepared ...,"Fangjun Hu, Christian Kokail, Milan Kornjača, ...",2026-04-01T17:42:56Z,2026-06-15T18:22:34Z,2026,quant-ph,"quant-ph, cond-mat.stat-mech, cs.CC, cs.LG",Learning quantum states from measurement data ...,"44 pages, 14 figures, 1 table",,,https://arxiv.org/abs/2604.01197v4,https://arxiv.org/pdf/2604.01197v4,22,quantum generative model
9,"all:""quantum diffusion model""",2603.06488v3,2603.06488,Score Reversal Is Not Free for Qu

## 6. Filtragem dos resultados mais relevantes

A pontuação é heurística. Ela ajuda a priorizar papers realmente ligados a modelos de difusão/generativos, mas recomenda-se conferir manualmente título e resumo.


In [ ]:
if df_all.empty:
    df_filtered = df_all.copy()
else:
    df_filtered = df_all[df_all["relevance_score"] >= MIN_RELEVANCE_SCORE].copy()
    df_filtered = df_filtered.sort_values(["relevance_score", "published"], ascending=[False, False])

csv_filtered = OUTPUT_DIR / "arxiv_quantum_diffusion_filtered.csv"
xlsx_filtered = OUTPUT_DIR / "arxiv_quantum_diffusion_filtered.xlsx"

df_filtered.to_csv(csv_filtered, index=False)
df_filtered.to_excel(xlsx_filtered, index=False)

print(f"Total filtrado: {len(df_filtered)}")
print(f"CSV filtrado salvo em: {csv_filtered}")
print(f"XLSX filtrado salvo em: {xlsx_filtered}")

df_filtered[[
    "relevance_score", "year", "topic_class", "title", "authors", "arxiv_id_base", "primary_category", "abs_url"
]].head(30)


Total filtrado: 58
CSV filtrado salvo em: arxiv_quantum_diffusion_results/arxiv_quantum_diffusion_filtered.csv
XLSX filtrado salvo em: arxiv_quantum_diffusion_results/arxiv_quantum_diffusion_filtered.xlsx


,relevance_score,year,topic_class,title,authors,arxiv_id_base,primary_category,abs_url
40,38,2024,DDPM / denoising diffusion,Mixed-State Quantum Denoising Diffusion Probab...,"Gino Kwun, Bingzhi Zhang, Quntao Zhuang",2411.17608,quant-ph,https://arxiv.org/abs/2411.17608v2
11,37,2026,DDPM / denoising diffusion,Learning Quantum Data Distribution via Chaotic...,"Quoc Hoan Tran, Koki Chinzei, Yasuhiro Endo, H...",2602.22061,quant-ph,https://arxiv.org/abs/2602.22061v2
16,37,2025,DDPM / denoising diffusion,Mitigating Barren Plateaus in Quantum Denoisin...,"Haipeng Cao, Kaining Zhang, Dacheng Tao, Zhaof...",2512.06695,cs.LG,https://arxiv.org/abs/2512.06695v2
20,36,2025,DDPM / denoising diffusion,DiffQ: Unified Parameter Initialization for Va...,"Chi Zhang, Mengxin Zheng, Qian Lou, Fan Chen",2509.17324,cs.ET,https://arxiv.org/abs/2509.17324v1
53,35,2023,DDPM / denoising diffusion,Generative quantum machine learning via denois...,"Bingzhi Zhang, Peng Xu, Xiaohui Chen, Quntao Z...",2310.05866,quant-ph,https://arxiv.org/abs/2310.05866v5
4,34,2026,score-based diffusion,Stochastic Schrödinger Diffusion Models for Pu...,"Jian Xu, Wei Chen, Shigui Li, Chao Li, Jingyua...",2605.03573,stat.ML,https://arxiv.org/abs/2605.03573v3
10,34,2026,DDPM / denoising diffusion,Identification of quantum generative circuits ...,"Zheping Wu, Xiaopeng Huang, Hengyue Jia, Haobi...",2603.02834,quant-ph,https://arxiv.org/abs/2603.02834v1
33,34,2025,DDPM / denoising diffusion,Overcoming Dimensional Factorization Limits in...,"Chuangtao Chen, Qinglin Zhao, MengChu Zhou, Du...",2505.05151,quant-ph,https://arxiv.org/abs/2505.05151v3
45,33,2024,DDPM / denoising diffusion,Conditional Diffusion-based Parameter Generati...,"Fanxu Meng, Xiangzhen Zhou, Pengcheng Zhu, Yu Luo",2407.12242,quant-ph,https://arxiv.org/abs/2407.12242v4
32,30,2025,DDPM / denoising diffusion,Leveraging Diffusion Models for Parameterized ...,"Daniel Barta, Darya Martyniuk, Johannes Jung, ...",2505.20863,quant-ph,https://arxiv.org/abs/2505.20863v3


## 7. Tabelas-resumo

Estas tabelas ajudam a enxergar rapidamente quais anos, categorias e tópicos aparecem mais.


In [ ]:
if not df_filtered.empty:
    display(df_filtered["year"].value_counts().sort_index().rename_axis("year").reset_index(name="n_papers"))
    display(df_filtered["primary_category"].value_counts().rename_axis("primary_category").reset_index(name="n_papers").head(20))
    display(df_filtered["topic_class"].value_counts().rename_axis("topic_class").reset_index(name="n_papers"))
else:
    print("Nenhum resultado filtrado. Tente reduzir MIN_RELEVANCE_SCORE ou MIN_YEAR.")


,year,n_papers
0,2020,1
1,2022,1
2,2023,6
3,2024,11
4,2025,25
5,2026,14


,primary_category,n_papers
0,quant-ph,39
1,cs.LG,4
2,nucl-th,4
3,math.OC,3
4,stat.ML,1
5,cs.ET,1
6,cs.CV,1
7,hep-ex,1
8,hep-lat,1
9,math-ph,1


,topic_class,n_papers
0,DDPM / denoising diffusion,17
1,quantum circuit / QNN generative model,17
2,other / needs manual check,9
3,score-based diffusion,6
4,quantum generative model,6
5,Schrödinger bridge,3


## 8. Busca manual dentro dos resultados

Use esta célula para procurar termos específicos dentro dos resultados baixados.


In [ ]:
def search_inside_results(df: pd.DataFrame, terms: str, top_k: int = 30) -> pd.DataFrame:
    """Filtra resultados que contenham todos os termos informados."""
    if df.empty:
        return df

    tokens = [t.strip().lower() for t in re.split(r"\s+", terms) if t.strip()]
    text = (df["title"].fillna("") + " " + df["summary"].fillna("")).str.lower()

    mask = pd.Series(True, index=df.index)
    for token in tokens:
        mask &= text.str.contains(re.escape(token), na=False)

    cols = ["relevance_score", "year", "topic_class", "title", "authors", "arxiv_id_base", "abs_url", "summary"]
    return df.loc[mask, cols].head(top_k)

# Exemplos:
# search_inside_results(df_all, "quantum score diffusion")
# search_inside_results(df_all, "denoising diffusion")
# search_inside_results(df_all, "stochastic schrodinger")

search_inside_results(df_all, "quantum diffusion")


,relevance_score,year,topic_class,title,authors,arxiv_id_base,abs_url,summary
0,18,2026,score-based diffusion,The Score Hamiltonian: Mapping Diffusion Model...,"Peter Halmos, Boris Hanin",2606.05217,https://arxiv.org/abs/2606.05217v1,We exhibit an exact correspondence between sam...
1,18,2026,quantum circuit / QNN generative model,Hybrid Quantum-Classical Corrective Diffusion ...,"Rui Wang, Edoardo Pasetto, Amer Delilbasic, Mo...",2605.23403,https://arxiv.org/abs/2605.23403v1,Statistical downscaling is a crucial component...
2,21,2026,DDPM / denoising diffusion,Hypothesis Tests for Observing Quantum Entangl...,"Vincent Alexander Croft, Lennart Voelz, Andrii...",2605.19754,https://arxiv.org/abs/2605.19754v1,We present a novel experimental strategy for t...
3,18,2026,DDPM / denoising diffusion,Physics Guided Generative Optimization for Tro...,WenBin Yan,2605.13268,https://arxiv.org/abs/2605.13268v2,Trotter Suzuki product formulas are the standa...
4,34,2026,score-based diffusion,Stochastic Schrödinger Diffusion Models for Pu...,"Jian Xu, Wei Chen, Shigui Li, Chao Li, Jingyua...",2605.03573,https://arxiv.org/abs/2605.03573v3,Quantum machine learning increasingly relies o...
5,19,2026,quantum circuit / QNN generative model,From Characterization To Construction: Generat...,"King Yiu Yu, Aritra Sarkar, Erbing Hua, Maximi...",2605.01367,https://arxiv.org/abs/2605.01367v1,High-fidelity circuit execution on noisy inter...
6,23,2026,score-based diffusion,The Feedback Hamiltonian is the Score Function...,"Sagar Dubey, Alan John",2604.21210,https://arxiv.org/abs/2604.21210v1,"In continuously monitored quantum systems, the..."
7,12,2026,other / needs manual check,Deuteron normalization and channel-dependent f...,"Byung-Geel Yu, Kook-Jin Kong, Tae Keun Choi",2604.05612,https://arxiv.org/abs/2604.05612v2,A combined view of the Jefferson Lab data on n...
8,22,2026,quantum generative model,Learning and Generating Mixed States Prepared ...,"Fangjun Hu, Christian Kokail, Milan Kornjača, ...",2604.01197,https://arxiv.org/abs/2604.01197v4,Learning quantum states from measurement data ...
9,12,2026,other / needs manual check,Score Reversal Is Not Free for Quantum Diffusi...,Ammar Fayad,2603.06488,https://arxiv.org/abs/2603.06488v3,Classical reverse diffusion is generated by ch...


## 9. Exportar lista em Markdown para usar no texto/projeto

Gera uma lista curta com título, ano, autores, arXiv ID e links.


In [ ]:
def short_authors(authors: str, max_authors: int = 4) -> str:
    names = [a.strip() for a in str(authors).split(",") if a.strip()]
    if len(names) <= max_authors:
        return ", ".join(names)
    return ", ".join(names[:max_authors]) + ", et al."


def make_markdown_bibliography(df: pd.DataFrame, top_k: int = 60) -> str:
    lines = []
    for _, row in df.head(top_k).iterrows():
        title = row.get("title", "")
        year = row.get("year", "")
        authors = short_authors(row.get("authors", ""))
        arxiv_id = row.get("arxiv_id_base", "")
        abs_url = row.get("abs_url", "")
        pdf_url = row.get("pdf_url", "")
        topic = row.get("topic_class", "")
        score = row.get("relevance_score", "")

        lines.append(
            f"- **{title}** ({year}). {authors}. "
            f"arXiv:{arxiv_id}. Tema: {topic}. Score: {score}. "
            f"[abs]({abs_url}) | [pdf]({pdf_url})"
        )
    return "\n".join(lines)

markdown_text = make_markdown_bibliography(df_filtered, top_k=60)

md_path = OUTPUT_DIR / "arxiv_quantum_diffusion_bibliography.md"
md_path.write_text(markdown_text, encoding="utf-8")

print(f"Markdown salvo em: {md_path}")
print(markdown_text[:4000])


Markdown salvo em: arxiv_quantum_diffusion_results/arxiv_quantum_diffusion_bibliography.md
- **Mixed-State Quantum Denoising Diffusion Probabilistic Model** (2024). Gino Kwun, Bingzhi Zhang, Quntao Zhuang. arXiv:2411.17608. Tema: DDPM / denoising diffusion. Score: 38. [abs](https://arxiv.org/abs/2411.17608v2) | [pdf](https://arxiv.org/pdf/2411.17608v2)
- **Learning Quantum Data Distribution via Chaotic Quantum Diffusion Model** (2026). Quoc Hoan Tran, Koki Chinzei, Yasuhiro Endo, Hirotaka Oshima. arXiv:2602.22061. Tema: DDPM / denoising diffusion. Score: 37. [abs](https://arxiv.org/abs/2602.22061v2) | [pdf](https://arxiv.org/pdf/2602.22061v2)
- **Mitigating Barren Plateaus in Quantum Denoising Diffusion Probabilistic Model** (2025). Haipeng Cao, Kaining Zhang, Dacheng Tao, Zhaofeng Su. arXiv:2512.06695. Tema: DDPM / denoising diffusion. Score: 37. [abs](https://arxiv.org/abs/2512.06695v2) | [pdf](https://arxiv.org/pdf/2512.06695v2)
- **DiffQ: Unified Parameter Initialization for Variat

## 10. Opcional: baixar PDFs dos artigos mais relevantes

Use com cuidado para não baixar muitos arquivos de uma vez.


In [ ]:
def download_top_pdfs(
    df: pd.DataFrame,
    top_k: int = 10,
    output_dir: Path = OUTPUT_DIR / "pdfs",
    sleep_seconds: float = 3.0,
):
    output_dir.mkdir(parents=True, exist_ok=True)
    downloaded = []

    for _, row in tqdm(df.head(top_k).iterrows(), total=min(top_k, len(df)), desc="Baixando PDFs"):
        arxiv_id = row.get("arxiv_id_base", "")
        title = row.get("title", "paper")
        pdf_url = row.get("pdf_url", "") or f"https://arxiv.org/pdf/{arxiv_id}"
        filename = f"{arxiv_id}_{safe_filename(title)}.pdf"
        path = output_dir / filename

        if path.exists():
            downloaded.append(path)
            continue

        response = requests.get(pdf_url, timeout=90)
        response.raise_for_status()
        path.write_bytes(response.content)
        downloaded.append(path)
        time.sleep(sleep_seconds)

    return downloaded

# Exemplo de uso:
# downloaded_files = download_top_pdfs(df_filtered, top_k=5)
# downloaded_files
